# Notebook de Desarrollo 1
David Díaz Sánchez

# Carga y limpieza de Raw Data — KS_NFA_FTNESS (medición de condición física)

Pipeline que:
1. Carga y concatena todos los CSV mensuales.
2. Traduce columnas y valores categóricos (KOR → EN).
3. Convierte tipos, valida los mapeos y detecta valores fuera de rango.
4. Filtra al grupo objetivo, deriva `vo2max_estimate` y elimina columnas redundantes/con exceso de nulos.
5. Guarda el dataset limpio en `Data/processed/`.


Link de descarga de los datos: 
https://www.bigdata-culture.kr/bigdata/user/data_market/detail.do?id=ace0aea7-5eee-48b9-b616-637365d665c1

## 0. Configuración

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- Rutas -------------------------------------------------------------
RAW_DIR = Path("../Data/raw_data")
OUTPUT_DIR = Path("../Data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Parámetros del pipeline --------------------------------------------
ENCODINGS_TO_TRY = ["utf-8-sig"]  # cp949: encoding típico de datos coreanos
TARGET_AGE_GROUPS = ["adult"]
NULL_THRESHOLD = 90.0  # % de nulos a partir del cual una columna se elimina (tras filtrar por edad adulta) no aportan info relevante

# Rangos fisiológicamente posibles: usados solo para identificar posibles outliers
PLAUSIBLE_RANGES = {
    "age": (0, 110),
    "height_cm": (50, 230),
    "weight_kg": (10, 300),
    "bmi": (10, 70),
    "systolic_bp": (60, 260),
    "diastolic_bp": (30, 160),
}

pd.set_option("display.max_rows", 100)

## 1. Carga y concatenación de todos los meses disponibles

In [2]:
def read_csv_robust(path: Path, encodings=ENCODINGS_TO_TRY) -> tuple[pd.DataFrame, str]:
    """Intenta leer un CSV probando encodings en orden; devuelve (df, encoding_usado)."""
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False), enc
        except UnicodeDecodeError as e:
            last_error = e
    raise UnicodeDecodeError(
        f"No se pudo leer {path.name} con ninguno de los encodings probados {encodings}"
    ) from last_error


files = sorted(RAW_DIR.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No se encontraron CSVs en {RAW_DIR.resolve()}")

print(f"{len(files)} archivos encontrados en {RAW_DIR}")

40 archivos encontrados en ../Data/raw_data


In [3]:
dfs = []
fallback_used = []  # archivos que tuvieron issues con el encoding

for f in tqdm(files, desc="Leyendo CSVs"):
    df, encoding_used = read_csv_robust(f)
    if encoding_used != ENCODINGS_TO_TRY[0]:
        fallback_used.append((f.name, encoding_used))
    dfs.append(df)

print(f"\nTotal filas leídas: {sum(len(d) for d in dfs):,}")
if fallback_used:
    print(f"\n⚠️  {len(fallback_used)} archivo(s) requirieron un encoding de respaldo (revisar por posible corrupción):")
    for name, enc in fallback_used:
        print(f"  - {name}: {enc}")

Leyendo CSVs:   0%|          | 0/40 [00:00<?, ?it/s]


Total filas leídas: 983,607


In [4]:
# ---------------------------------------------------------------------------
# Validación de esquema: confirmar que todos tengan el mismo layout para no 
# generar columnas mal por concatenación automatica
# ---------------------------------------------------------------------------
reference_cols = frozenset(dfs[0].columns)
schema_mismatches = {
    f.name: {
        "faltantes": sorted(reference_cols - frozenset(d.columns)),
        "extra": sorted(frozenset(d.columns) - reference_cols),
    }
    for f, d in zip(files, dfs)
    if frozenset(d.columns) != reference_cols
}

if schema_mismatches:
    print("⚠️  Archivos con esquema distinto al de referencia:")
    for name, diff in schema_mismatches.items():
        print(f"  - {name}: faltantes={diff['faltantes']} extra={diff['extra']}")
    raise ValueError(
        "Hay archivos con columnas distintas al resto. Revísalos antes de concatenar "
        "(un concat silencioso generaría NaNs disfrazados de 'missing values' reales)."
    )

print("✅ Todos los archivos comparten el mismo esquema. Concatenando...")
raw = pd.concat(dfs, ignore_index=True)
print(f"Dataset combinado: {raw.shape[0]:,} filas, {raw.shape[1]} columnas")

✅ Todos los archivos comparten el mismo esquema. Concatenando...
Dataset combinado: 983,607 filas, 55 columnas


## 2. Traducción de columnas (basado en el diccionario oficial)

In [5]:
# Diccionario de traducción y significado de las columnas/variables del Dataset 
column_rename = {
    "MESURE_TME": "measurement_round",
    "CNTER_NM": "center_name",
    "AGE_FLAG_NM": "age_group",                       # clasificación por edad (niñez/adulto/adulto mayor)
    "MESURE_PLACE_FLAG_NM": "measurement_location_type",
    "MESURE_AGE_CO": "age",
    "INPT_FLAG_NM": "input_type",
    "COAW_FLAG_NM": "listing_type",
    "MESURE_DAY": "measurement_date",
    "SEXDSTN_FLAG_CD": "gender_code",
    "MESURE_IEM_001_VALUE": "height_cm",
    "MESURE_IEM_002_VALUE": "weight_kg",
    "MESURE_IEM_003_VALUE": "body_fat_pct",
    "MESURE_IEM_004_VALUE": "waist_circumference_cm",
    "MESURE_IEM_005_VALUE": "diastolic_bp",
    "MESURE_IEM_006_VALUE": "systolic_bp",
    "MESURE_IEM_007_VALUE": "grip_strength_left_kg",
    "MESURE_IEM_008_VALUE": "grip_strength_right_kg",
    "MESURE_IEM_009_VALUE": "situps_count",
    "MESURE_IEM_010_VALUE": "repeated_jump_count",
    "MESURE_IEM_012_VALUE": "sit_and_reach_cm",
    "MESURE_IEM_013_VALUE": "illinois_agility_sec",
    "MESURE_IEM_014_VALUE": "endurance_sec",
    "MESURE_IEM_015_VALUE": "coordination_time_sec",
    "MESURE_IEM_016_VALUE": "coordination_errors_count",
    "MESURE_IEM_017_VALUE": "coordination_result_sec",
    "MESURE_IEM_018_VALUE": "bmi",
    "MESURE_IEM_019_VALUE": "cross_situp_count",
    "MESURE_IEM_020_VALUE": "long_roundtrip_count",
    "MESURE_IEM_021_VALUE": "roundtrip_10m_4_sec",
    "MESURE_IEM_022_VALUE": "standing_long_jump_cm",
    "MESURE_IEM_023_VALUE": "chair_situp_count",
    "MESURE_IEM_024_VALUE": "walk_6min_m",
    "MESURE_IEM_025_VALUE": "walk_in_place_2min_count",
    "MESURE_IEM_026_VALUE": "chair_3m_target_return_sec",
    "MESURE_IEM_027_VALUE": "walk_8way_count",
    "MESURE_IEM_028_VALUE": "relative_grip_strength_pct",
    "MESURE_IEM_029_VALUE": "double_skinfold",
    "MESURE_IEM_030_VALUE": "long_roundtrip_vo2max",
    "MESURE_IEM_031_VALUE": "treadmill_resting_bpm",
    "MESURE_IEM_032_VALUE": "treadmill_3min_bpm",
    "MESURE_IEM_033_VALUE": "treadmill_6min_bpm",
    "MESURE_IEM_034_VALUE": "treadmill_9min_bpm",
    "MESURE_IEM_035_VALUE": "treadmill_vo2max",
    "MESURE_IEM_036_VALUE": "step_test_recovery_bpm",
    "MESURE_IEM_037_VALUE": "step_test_vo2max",
    "MESURE_IEM_038_VALUE": "thigh_left_cm",
    "MESURE_IEM_039_VALUE": "thigh_right_cm",
    "MESURE_IEM_040_VALUE": "reaction_time_sec",
    "MESURE_IEM_041_VALUE": "adult_endurance_sec",
    "MESURE_IEM_042_VALUE": "waist_to_height_ratio",
    "MESURE_IEM_043_VALUE": "repeated_side_jump_count",
    "MESURE_IEM_044_VALUE": "eye_hand_coordination_sec",
    "MESURE_IEM_050_VALUE": "roundtrip_5m_4_sec",
    "MESURE_IEM_051_VALUE": "button_press_3x3_sec",
    "MESURE_IEM_052_VALUE": "absolute_grip_strength_kg",
}

unmapped = set(raw.columns) - set(column_rename)
if unmapped:
    raise KeyError(f"Columnas del CSV sin entrada en column_rename: {sorted(unmapped)}")

raw = raw.rename(columns=column_rename)

## 3. Conversión de tipos


In [6]:
categorical_cols = [
    "center_name", "age_group", "measurement_location_type",
    "input_type", "listing_type", "gender_code",]

numeric_cols = [c for c in raw.columns if c not in categorical_cols + ["measurement_date"]]

raw["measurement_date"] = pd.to_datetime(raw["measurement_date"], format="%Y%m%d", errors="coerce")
for col in categorical_cols:
    raw[col] = raw[col].astype("category")

coercion_report = {}
for col in numeric_cols:
    non_null_before = raw[col].notna().sum()
    raw[col] = pd.to_numeric(raw[col], errors="coerce")
    non_null_after = raw[col].notna().sum()
    lost = non_null_before - non_null_after
    if lost > 0:
        coercion_report[col] = lost

if coercion_report:
    print("⚠️  Valores no numéricos convertidos a NaN al forzar el tipo (posibles errores de captura):")
    for col, n in sorted(coercion_report.items(), key=lambda kv: -kv[1]):
        print(f"  - {col}: {n} valor(es)")
else:
    print("✅ Ninguna columna numérica contenía valores no numéricos.")

n_bad_dates = raw["measurement_date"].isna().sum()
if n_bad_dates:
    print(f"⚠️  {n_bad_dates} fecha(s) no se pudieron parsear con formato YYYYMMDD.")

⚠️  Valores no numéricos convertidos a NaN al forzar el tipo (posibles errores de captura):
  - systolic_bp: 7 valor(es)
  - diastolic_bp: 2 valor(es)
  - grip_strength_left_kg: 1 valor(es)
  - coordination_errors_count: 1 valor(es)


## 4. Mapeo de valores categóricos a inglés

`gender_code` ya viene en inglés (`'F'` / `'M'`) — se valida en vez de asumirlo. Para `age_group` y
`measurement_location_type`, `.map()` convierte en `NaN` cualquier valor no contemplado en el
diccionario, cada mapeo se valida comparando nulos.


In [7]:
def map_and_validate(series: pd.Series, mapping: dict, name: str) -> pd.Series:
    nulls_before = series.isna().sum()
    mapped = series.map(mapping)
    nulls_after = mapped.isna().sum()
    newly_null = nulls_after - nulls_before
    if newly_null > 0:
        unmapped_values = sorted(set(series.dropna().unique()) - set(mapping))
        raise ValueError(
            f"{name}: {newly_null} valor(es) no estaban en el diccionario de mapeo: {unmapped_values}"
        )
    return mapped


gender_allowed = {"F", "M"}
gender_unexpected = set(raw["gender_code"].dropna().unique()) - gender_allowed


if gender_unexpected:
    raise ValueError(f"gender_code contiene valores inesperados: {gender_unexpected}")

age_group_map = {
    "유아기": "early_childhood",   # primera infancia (preescolar)
    "유소년": "child",             # niño / preadolescente
    "청소년": "adolescent",        # adolescente
    "성인": "adult",               # adulto
    "어르신": "senior",            # adulto mayor
}

raw["age_group"] = map_and_validate(raw["age_group"], age_group_map, "age_group")

measurement_location_map = {
    "일반": "standard",   # medición en el centro
    "출장": "outreach",   # medición itinerante (equipo desplazado)
}

raw["measurement_location_type"] = map_and_validate(
    raw["measurement_location_type"], measurement_location_map, "measurement_location_type"
)

# age_group es ordinal (hay un orden natural de edad), no nominal:
age_order = ["early_childhood", "child", "adolescent", "adult", "senior"]
raw["age_group"] = pd.Categorical(raw["age_group"], categories=age_order, ordered=True)

print("Valores únicos en 'age_group':", raw["age_group"].unique().tolist())
print("Valores únicos en 'measurement_location_type':", raw["measurement_location_type"].unique().tolist())

Valores únicos en 'age_group': ['senior', 'adolescent', 'adult', 'child', 'early_childhood']
Valores únicos en 'measurement_location_type': ['standard', 'outreach']


## 5. Columnas de metadata administrativa

`center_name`, `input_type`, `listing_type` describen el centro/proceso de captura, no la condición
física de la persona — se eliminan ya que no se van a usar como variable de análisis.


In [8]:
drop_cols = ["center_name", "input_type", "listing_type"]
raw_clean = raw.drop(columns=[c for c in drop_cols if c in raw.columns]).copy()

## 6. Nulos por columna

Muchas pruebas (treadmill, step test, caminata de 6 min, etc.) solo aplican a ciertos grupos de
edad. Es normal que la mayoría de filas tengan `NaN` en columnas que no les correspondían. En vez de
eliminar filas con `NaN`, se segmenta por `age_group` y se usan solo las columnas relevantes para
cada grupo.


In [9]:
def null_report(df: pd.DataFrame, label: str) -> pd.Series:
    pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
    print(f"\n% de nulos por columna — {label}:")
    print(pct)
    return pct


_ = null_report(raw_clean, "antes de filtrar por edad")


% de nulos por columna — antes de filtrar por edad:
thigh_left_cm                 100.0
thigh_right_cm                100.0
double_skinfold               100.0
walk_6min_m                    99.9
treadmill_9min_bpm             97.9
button_press_3x3_sec           96.4
roundtrip_5m_4_sec             96.4
treadmill_resting_bpm          92.5
treadmill_vo2max               92.5
treadmill_6min_bpm             92.5
treadmill_3min_bpm             92.5
roundtrip_10m_4_sec            90.3
absolute_grip_strength_kg      90.0
walk_in_place_2min_count       89.2
chair_3m_target_return_sec     89.1
chair_situp_count              89.0
walk_8way_count                88.8
repeated_side_jump_count       87.4
eye_hand_coordination_sec      87.4
waist_to_height_ratio          87.3
adult_endurance_sec            85.4
repeated_jump_count            84.0
step_test_recovery_bpm         75.7
step_test_vo2max               75.7
endurance_sec                  72.4
coordination_time_sec          72.1
coordinatio

## 7. Filtrado al grupo objetivo

Nos quedamos unicamente con los datos para adultos, ya que la app hará enfoque en este segmento

In [10]:
print(f"Filas antes de filtrar por edad: {raw_clean.shape[0]:,}")
raw_clean = raw_clean[raw_clean["age_group"].isin(TARGET_AGE_GROUPS)].reset_index(drop=True)
raw_clean["age_group"] = raw_clean["age_group"].cat.remove_unused_categories()
print(f"Filas después de filtrar solo {TARGET_AGE_GROUPS}: {raw_clean.shape[0]:,}")

_ = null_report(raw_clean, f"ya filtrado a {TARGET_AGE_GROUPS}")

Filas antes de filtrar por edad: 983,607
Filas después de filtrar solo ['adult']: 430,666

% de nulos por columna — ya filtrado a ['adult']:
coordination_time_sec         100.0
situps_count                  100.0
endurance_sec                 100.0
illinois_agility_sec          100.0
repeated_jump_count           100.0
thigh_right_cm                100.0
absolute_grip_strength_kg     100.0
button_press_3x3_sec          100.0
eye_hand_coordination_sec     100.0
roundtrip_5m_4_sec            100.0
waist_to_height_ratio         100.0
repeated_side_jump_count      100.0
walk_6min_m                   100.0
walk_in_place_2min_count      100.0
chair_3m_target_return_sec    100.0
walk_8way_count               100.0
double_skinfold               100.0
chair_situp_count             100.0
coordination_result_sec       100.0
coordination_errors_count     100.0
thigh_left_cm                 100.0
treadmill_9min_bpm             95.3
treadmill_6min_bpm             84.9
treadmill_3min_bpm             

## 8. Derivar `vo2max_estimate`

Combina las tres estimaciones alternativas de VO2 máx (treadmill / step test / ida-y-vuelta larga)
en una sola columna, tomando la primera disponible por fila.


In [11]:
null_pct_before = raw_clean["treadmill_vo2max"].isna().mean() * 100

raw_clean["vo2max_estimate"] = (
    raw_clean["treadmill_vo2max"]
    .combine_first(raw_clean["step_test_vo2max"])
    .combine_first(raw_clean["long_roundtrip_vo2max"])
)

null_pct_after = raw_clean["vo2max_estimate"].isna().mean() * 100
print(f"% nulos solo con treadmill_vo2max: {null_pct_before:.1f}%")
print(f"% nulos en vo2max_estimate (combinando los 3 protocolos): {null_pct_after:.1f}%")

% nulos solo con treadmill_vo2max: 84.9%
% nulos en vo2max_estimate (combinando los 3 protocolos): 0.3%


## 9. Criterios para borrar columnas

**Criterio A** ya se extrajo la información útil de los 3 protocolos alternativos en
`vo2max_estimate`, así que sus columnas provenientes se desechan

**Criterio B (umbral):** columnas con alto porcentaje de nulos que no contienen información util o suficiente
para realizar un modelo.


In [12]:
redundant_vo2max_cols = [
    "treadmill_resting_bpm", "treadmill_3min_bpm", "treadmill_6min_bpm",
    "treadmill_9min_bpm", "treadmill_vo2max",
    "step_test_recovery_bpm", "step_test_vo2max",
    "long_roundtrip_vo2max", "long_roundtrip_count",
    "roundtrip_10m_4_sec", "adult_endurance_sec",]

raw_clean = raw_clean.drop(columns=[c for c in redundant_vo2max_cols if c in raw_clean.columns])

moderate_missing = raw_clean.isna().mean() * 100
print("Columnas con missing moderado que se conservan (se imputan/manejan después):")
print(moderate_missing[(moderate_missing > 5) & (moderate_missing < NULL_THRESHOLD)].round(1).sort_values(ascending=False))

Columnas con missing moderado que se conservan (se imputan/manejan después):
waist_circumference_cm    44.2
standing_long_jump_cm     35.7
reaction_time_sec         24.4
dtype: float64


In [13]:
null_pct = raw_clean.isna().mean() * 100
cols_to_drop = null_pct[null_pct >= NULL_THRESHOLD].index.tolist()
print(f"Columnas eliminadas por quedar en >={NULL_THRESHOLD}% de nulos: {cols_to_drop}")
raw_clean = raw_clean.drop(columns=cols_to_drop)

Columnas eliminadas por quedar en >=90.0% de nulos: ['situps_count', 'repeated_jump_count', 'illinois_agility_sec', 'endurance_sec', 'coordination_time_sec', 'coordination_errors_count', 'coordination_result_sec', 'chair_situp_count', 'walk_6min_m', 'walk_in_place_2min_count', 'chair_3m_target_return_sec', 'walk_8way_count', 'double_skinfold', 'thigh_left_cm', 'thigh_right_cm', 'waist_to_height_ratio', 'repeated_side_jump_count', 'eye_hand_coordination_sec', 'roundtrip_5m_4_sec', 'button_press_3x3_sec', 'absolute_grip_strength_kg']


## 10. Controles de calidad finales

- **Duplicados exactos**: filas idénticas en todas las columnas (posible doble carga de un mismo mes).
- **Rangos fisiológicamente plausibles**: solo se reportan, no se eliminan — decidir el tratamiento
  (revisar, capar o descartar).

In [14]:
n_dupes = raw_clean.duplicated().sum()
print(f"Filas duplicadas exactas: {n_dupes:,}")
if n_dupes:
    print("  (revisar si corresponden a un CSV cargado dos veces antes de eliminarlas)")

print("\nValores fuera de rango plausible (solo reporte, no se eliminan):")
for col, (low, high) in PLAUSIBLE_RANGES.items():
    if col not in raw_clean.columns:
        continue
    out_of_range = ~raw_clean[col].between(low, high) & raw_clean[col].notna()
    n_out = out_of_range.sum()
    if n_out:
        print(f"  - {col}: {n_out} valor(es) fuera de [{low}, {high}]")

Filas duplicadas exactas: 23
  (revisar si corresponden a un CSV cargado dos veces antes de eliminarlas)

Valores fuera de rango plausible (solo reporte, no se eliminan):
  - bmi: 11 valor(es) fuera de [10, 70]
  - systolic_bp: 1257 valor(es) fuera de [60, 260]
  - diastolic_bp: 1012 valor(es) fuera de [30, 160]


## 11. Resumen final y persistencia

In [15]:
# con meses de datos acumulados, esto reduce a la mitad la memoria de las columnas numéricas.

float_cols = raw_clean.select_dtypes(include="float64").columns
raw_clean[float_cols] = raw_clean[float_cols].astype("float32")

print(f"Shape final: {raw_clean.shape}")
print(f"Memoria: {raw_clean.memory_usage(deep=True).sum() / 1e6:.1f} MB")
raw_clean.info()

Shape final: (430666, 21)
Memoria: 37.5 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 430666 entries, 0 to 430665
Data columns (total 21 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   measurement_round           430666 non-null  int64         
 1   age_group                   430666 non-null  category      
 2   measurement_location_type   430666 non-null  category      
 3   age                         430666 non-null  int64         
 4   measurement_date            430666 non-null  datetime64[ns]
 5   gender_code                 430666 non-null  category      
 6   height_cm                   430645 non-null  float32       
 7   weight_kg                   430645 non-null  float32       
 8   body_fat_pct                428357 non-null  float32       
 9   waist_circumference_cm      240260 non-null  float32       
 10  diastolic_bp                430590 non-null  float32       
 

Guardamos el dataset limpio en un parquet listo para continuar con el proyecto

In [16]:
output_path = OUTPUT_DIR / "fitness_measurements_adult_clean.parquet"
raw_clean.to_parquet(output_path, index=False)
print(f"Guardado en: {output_path.resolve()}")

Guardado en: /home/david-diaz/ds-venv/Modulo_5/Data/processed/fitness_measurements_adult_clean.parquet
